In [85]:
import nflreadpy as nfl
import pandas as pd

# Load weekly player stats for 2019-2025
player_stats = nfl.load_player_stats(range(2019,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

print(df.shape)
print(df['position'].value_counts())
print(df['season'].unique())

(129812, 150)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving

In [86]:
# What positions exist and how many rows per position
print(df['position'].value_counts())

# Confirm season/week ranges
print(df['season'].unique())
print(df['week'].unique())

# Check for missing values in key fantasy-relevant columns
key_cols = ['player_display_name', 'position', 'team', 'opponent_team',
            'carries', 'targets', 'receptions', 'fantasy_points', 'fantasy_points_ppr']
print(df[key_cols].isnull().sum())

position
WR     17683
LB     15492
CB     13329
RB     11087
DE     10828
DT     10108
TE      8740
SAF     6311
QB      4690
DB      4478
K       3914
P       3865
OT      3147
OLB     3062
FS      2487
G       2129
S       1829
ILB     1531
MLB     1437
C       1077
NT       930
FB       748
LS       482
DL       232
OL        44
Name: count, dtype: int64
[2019 2020 2021 2022 2023 2024 2025]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
player_display_name    152
position               152
team                     0
opponent_team            0
carries                  0
targets                  0
receptions               0
fantasy_points           0
fantasy_points_ppr       0
dtype: int64


In [87]:
# Sorting dataset into players and seasons (chronologically)
df = df.sort_values(['player_id','season','week']).reset_index(drop=True)


In [88]:
# ----------- WR/TE Feature Engineering -----------

# Past 3 game averages
df['targets_avg_3'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_avg_3'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_avg_3'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['target_share_avg_3'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_air_yards_avg_3'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['air_yards_share_avg_3'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_td_avg_3'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_after_catch_avg_3'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['targets_avg_5'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_avg_5'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_avg_5'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['target_share_avg_5'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_air_yards_avg_5'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['air_yards_share_avg_5'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_td_avg_5'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_after_catch_avg_5'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['targets_trend'] = df['targets_avg_3'] - df['targets_avg_5']
df['target_share_trend'] = df['target_share_avg_3'] - df['target_share_avg_5']
df['rec_yards_trend'] = df['rec_yards_avg_3'] - df['rec_yards_avg_5']
df['rec_air_yards_trend'] = df['rec_air_yards_avg_3'] - df['rec_air_yards_avg_5']



In [89]:
# ----------- RB Feature Engineering -----------

df['opportunities'] = df['carries'] + df['targets']

# Past 3 game averages
df['carries_avg_3'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_yards_avg_3'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_tds_avg_3'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['opportunities_avg_3'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())


# Past 5 game averages
df['carries_avg_5'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_yards_avg_5'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_tds_avg_5'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['opportunities_avg_5'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['carries_trend'] = df['carries_avg_3'] - df['carries_avg_5']
df['rushing_yards_trend'] = df['rushing_yards_avg_3'] - df['rushing_yards_avg_5']
df['opportunities_trend'] = df['opportunities_avg_3'] - df['opportunities_avg_5']


In [90]:
# ----------- QB Feature Engineering -----------

# Past 3 game averages
df['completions_avg_3'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['attempts_avg_3'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_tds_avg_3'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_int_avg_3'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_air_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_first_downs_avg_3'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['completions_avg_5'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['attempts_avg_5'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_tds_avg_5'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_int_avg_5'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_air_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_first_downs_avg_5'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['attempts_trend'] = df['attempts_avg_3'] - df['attempts_avg_5']
df['passing_yards_trend'] = df['passing_yards_avg_3'] - df['passing_yards_avg_5']
df['passing_air_yards_trend'] = df['passing_air_yards_avg_3'] - df['passing_air_yards_avg_5']


In [91]:
# ----------- K Feature Engineering -----------

# Past 3 game averages
df['fg_att_avg_3'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_avg_3'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_long_avg_3'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_50_59_avg_3'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_att_avg_3'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_made_avg_3'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['fg_att_avg_5'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_avg_5'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_long_avg_5'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_50_59_avg_5'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_att_avg_5'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_made_avg_5'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['fg_att_trend'] = df['fg_att_avg_3'] - df['fg_att_avg_5']
df['fg_made_trend'] = df['fg_made_avg_3'] - df['fg_made_avg_5']

In [95]:
# ----------- WR/TE Training Dataset -----------

wr_te_df = df[df['position'].isin(['WR', 'TE'])].copy()
wr_te_features = [
    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_air_yards_avg_3',
    'rec_air_yards_avg_5',
    'air_yards_share_avg_3',
    'air_yards_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',
    'rec_yards_after_catch_avg_3',
    'rec_yards_after_catch_avg_5',

    # Trends
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend',
    'rec_air_yards_trend'
]

X_wr_te = wr_te_df[wr_te_features]
y_wr_te = wr_te_df['fantasy_points_ppr']

# Removing empty row that has no prior game history
valid_wr_te = X_wr_te.notna().all(axis=1)

X_wr_te = X_wr_te[valid_wr_te]
y_wr_te = y_wr_te[valid_wr_te]


(23859, 20)
(23859,)
0


In [96]:
# ----------- RB Training Dataset -----------

rb_df = df[df['position'] == 'RB'].copy()
rb_features = [
    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',
    'opportunities_avg_3',
    'opportunities_avg_5',

    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',

    # Trends
    'carries_trend',
    'rushing_yards_trend',
    'opportunities_trend',
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend'
]

X_rb = rb_df[rb_features]
y_rb = rb_df['fantasy_points_ppr']

# Removing empty row that has no prior game history
valid_rb = X_rb.notna().all(axis=1)

X_rb = X_rb[valid_rb]
y_rb = y_rb[valid_rb]


In [ ]:
# ----------- QB Training Dataset -----------

qb_df = df[df['position'] == 'QB'].copy()

qb_features = [
    # Passing
    'completions_avg_3',
    'completions_avg_5',
    'attempts_avg_3',
    'attempts_avg_5',
    'passing_yards_avg_3',
    'passing_yards_avg_5',
    'passing_tds_avg_3',
    'passing_tds_avg_5',
    'passing_int_avg_3',
    'passing_int_avg_5',
    'passing_air_yards_avg_3',
    'passing_air_yards_avg_5',
    'passing_first_downs_avg_3',
    'passing_first_downs_avg_5',

    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',

    # Trends
    'attempts_trend',
    'passing_yards_trend',
    'passing_air_yards_trend',
    'carries_trend',
    'rushing_yards_trend'
]

X_qb = qb_df[qb_features]
y_qb = qb_df['fantasy_points_ppr']

valid_qb = X_qb.notna().all(axis=1)

X_qb = X_qb[valid_qb]
y_qb = y_qb[valid_qb]

In [100]:
# ----------- K Training Dataset -----------

k_df = df[df['position'] == 'K'].copy()

k_features = [
    'fg_att_avg_3',
    'fg_att_avg_5',
    'fg_made_avg_3',
    'fg_made_avg_5',
    'fg_long_avg_3',
    'fg_long_avg_5',
    'fg_made_50_59_avg_3',
    'fg_made_50_59_avg_5',
    'pat_att_avg_3',
    'pat_att_avg_5',
    'pat_made_avg_3',
    'pat_made_avg_5',
    'fg_att_trend',
    'fg_made_trend'
]

k_df['kicker_fantasy_points'] = (
    3 * k_df['fg_made'] +
    1 * k_df['pat_made']
)

X_k = k_df[k_features]
y_k = k_df['kicker_fantasy_points']

valid_k = X_k.notna().all(axis=1)

X_k = X_k[valid_k]
y_k = y_k[valid_k]
